Separable 6D convolutionn for 3D I/O system

---

Kishore Kumar Tarafdar, Date: 06-06-2025


In [1]:
pwd

'/data1/kishoretarafdar/src.port/NSLI.v00'

In [2]:
!python --version

Python 3.12.7


        Disable GPU: Force tensorflow to select CPU

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="-1"    
import tensorflow as tf

2025-06-29 09:36:28.499550: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751169988.521328 1254039 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751169988.528082 1254039 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-29 09:36:28.551442: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


        Select a GPU with a memory limit

In [3]:
import tensorflow as tf
print(f"TensorFlow version {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
len(gpus)

2025-06-06 01:42:07.550373: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749154327.573181 3080753 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749154327.580244 3080753 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-06 01:42:07.604417: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version 2.18.0
Num GPUs Available:  3


3

Select one GPU

        Restrict code to use a particular GPU...

In [4]:
# # include ../dirx 
mylibpath = [
    '/home/kishoretarafdar/bin',
    '/data1/kishoretarafdar/src.port/NSLI.v00/utils.Volterra'
    #'/home/k/PLAYGROUND10GB/SKULSTRIPpaper__'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath

from tf_select_a_gpu import select_a_gpu

In [5]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [6]:
# select_gpu = gpus[gpu_id]
memory_limit = 48#GB
select_a_gpu(gpus, gpu_id=2, memory_limit=memory_limit)
# del gpu_id, select_a_gpu, select_gpu

3 Physical GPUs available 
Selected 1 Logical GPU with 48 GB memory limit


I0000 00:00:1749154336.629758 3080753 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 49152 MB memory:  -> device: 2, name: NVIDIA RTX A6000, pci bus id: 0000:41:00.0, compute capability: 8.6


# Separable 6D convolution with 3D kernel

    Quadratic NLSI for 3D I/O




    
        Strategy tested with nonseparable conv2d 
        Perfect match with one channel input
        !! Does not match when multiple channel input

        !! Not possible to test the strategy with nonseparable high dimensional convolutions
        (apply update when a libray is located online for nonseparable 4d convolutions)

In [12]:
# import tensorflow as tf
# from tensorflow.keras.layers import Layer

# # include ../dirx 
mylibpath = [
    '/data1/kishoretarafdar/src.port/VolterraMRAsystems.v0/VolterraSys/ndconvolutions'
    ]
import sys
[sys.path.insert(0,_) for _ in mylibpath]
del mylibpath


import tensorflow as tf
from SeparableConvNDlayout import SeparableConvND


class SeparableConv6D(SeparableConvND):
    """Separable 6D convolution using 3D kernels

    VolterraSys: Multidimensional linear and nonlinear Volterra kernels in natural and multiresolution bases.
    Copyright (C) 2025 Kishore Kumar Tarafdar

    This program is free software: you can redistribute it and/or modify
    it under the terms of the GNU General Public License as published by
    the Free Software Foundation, either version 3 of the License, or
    (at your option) any later version.

    This program is distributed in the hope that it will be useful,
    but WITHOUT ANY WARRANTY; without even the implied warranty of
    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
    GNU General Public License for more details.

    You should have received a copy of the GNU General Public License
    along with this program.  If not, see <https://www.gnu.org/licenses/>.   
    
    --kkt@29-06-2025"""
    def __init__(self, filters, kernel=None, kernel_size=None, **kwargs):
        super().__init__(filters=filters, kernel=kernel, kernel_size=kernel_size, **kwargs)
        
    def call(self, inputs):
        return self.__separable_conv6d(inputs)


    def __separable_conv6d(self, x):
        # x: [B, N1, N2, N3, N4, N5, N6, C]
        input_shape = x.shape.as_list()
        N1, N2, N3, N4, N5, N6 = input_shape[1], input_shape[2], input_shape[3], input_shape[4], input_shape[5], input_shape[6]
        B = tf.shape(x)[0]

        # O = kernel3d.shape[-1]
        # h2 = kernel3d

        # Step 1: Convolve over (N1, N2, N3)
        x1 = tf.reshape(x, [-1, N4, N5, N6, self.inchannels])  # shape: (B*N3*N4, N1, N2, C)
        # x1 = tf.reshape(x, [B * N4 * N5 * N6, N1, N2, N3, self.inchannels])
        # print('x', x.shape)
        # print(x1.shape)
        x1 = tf.nn.convolution(x1, self.pointwise, padding='SAME')
        y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
        print('kernel', self.kernel.shape)
        # print(y1.shape)
        y1 = tf.reshape(y1, [B, N1, N2, N3, N4, N5, N6, self.filters])
        # print(y1.shape)
        y1 = tf.transpose(y1, perm=[0,4,5,6,1,2,3,7])
    
        # Step 2: Convolve over (N4, N5, N6)
        x2 = tf.reshape(y1, [-1, N1, N2, N3, self.filters])  # shape: (B*N1*N2, N3, N4, C) ## OK
        # x2 = tf.reshape(y1, [B * N1 * N2 * N3, N4, N5, N6, self.inchannels])
        # print(x2.shape)
        y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
        # print(y2.shape)
        y2 = tf.reshape(y2, [B, N4, N5, N6, N1, N2, N3, self.filters])
        # print(y2.shape)
        y2 = tf.transpose(y2, perm=[0,4,5,6,1,2,3,7])
        return y2
        

if __name__=='__main__':
    # Create a model using the layer
    inputs = tf.keras.Input(shape=(16, 16, 16, 16, 16, 16, 2))  # (N1, N2, N3, N4, C)
    # inputs = tf.keras.Input(shape=(16, 16, 16, 8, 8, 8, 2))  # (N1, N2, N3, N4, C)
    outputs = SeparableConv6D(filters=7, kernel_size=3)(inputs)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.summary()

    # # Test with random data
    test_input = tf.random.normal([2, 16, 16, 16, 16, 16, 16, 2])
    output = model(test_input)
    print(output.shape)
    del outputs, model, inputs, test_input, output



    ## example 2
    # Input dimensions
    B, N1, N2, N3, N4, N5, N6, C = 2, 8, 8, 8, 8, 8, 8, 3  # Batch size, 4D volume size, channels
    # B, N1, N2, N3, N4, C = 1, 2, 2, 2, 2, 1  # Batch size, 4D volume size, channels
    # B, N1, N2, N3, N4, C = 1, 3, 3, 2, 2, 1  # Batch size, 4D volume size, channels
    x = tf.random.normal((B, N1, N2, N3, N4, N5, N6, C))
    # x.shape
    layer = SeparableConv6D(filters=5, kernel_size=3)
    yout = layer(x)
    kernel3d = layer.get_kernel()
    print("Kernel shape:", kernel3d.shape)  # Should be (3, 3, 2, 32)
    print('yout', yout.shape)


kernel (3, 3, 3, 7, 7)


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 16, 16, 16, 16, │             0 │
│                                 │ 16, 16, 2)             │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv6d_12             │ (None, 16, 16, 16, 16, │         1,337 │
│ (SeparableConv6D)               │ 16, 16, 7)             │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,337 (5.22 KB)

 Trainable params: 1,323 (5.17 KB)

 Non-trainable params: 14 (56.00 B)

kernel (3, 3, 3, 7, 7)
(2, 16, 16, 16, 16, 16, 16, 7)
kernel (3, 3, 3, 5, 5)
Kernel shape: (3, 3, 3, 5, 5)
yout (2, 8, 8, 8, 8, 8, 8, 5)


In [12]:
# ## OK version 0
# import tensorflow as tf
# from tensorflow.keras.layers import Layer

# class SeparableConv6D(Layer):
#     """Separable 6D convolution using 3D kernels

#     VolterraSysMRA: Multidimensional linear and nonlinear Volterra kernels in natural and biortogonal bases.
#     Copyright (C) 2025 Kishore Kumar Tarafdar

#     This program is free software: you can redistribute it and/or modify
#     it under the terms of the GNU General Public License as published by
#     the Free Software Foundation, either version 3 of the License, or
#     (at your option) any later version.

#     This program is distributed in the hope that it will be useful,
#     but WITHOUT ANY WARRANTY; without even the implied warranty of
#     MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
#     GNU General Public License for more details.

#     You should have received a copy of the GNU General Public License
#     along with this program.  If not, see <https://www.gnu.org/licenses/>.   
    
#     @kkt 06-06-2025"""
#     def __init__(self, filters, kernel_size, **kwargs):
#         super(SeparableConv6D, self).__init__(**kwargs)
#         # self.filters = filters
#         self.kernel_size = kernel_size
        
#     def build(self, input_shape):
#         self.inchannels = input_shape[-1] 
#         self.filters = input_shape[-1]
#         # Create a 3D kernel that will be applied to both spatial dimensions
#         self.kernel = self.add_weight(
#             name='kernel3d',
#             shape=(self.kernel_size, self.kernel_size, self.kernel_size, input_shape[-1], self.filters),
#             initializer='glorot_uniform',
#             trainable=True
#         )
        
#     def call(self, inputs):
#         return self.__separable_conv6d(inputs)


#     def __separable_conv6d(self, x):
#         # x: [B, N1, N2, N3, N4, N5, N6, C]
#         input_shape = x.shape.as_list()
#         N1, N2, N3, N4, N5, N6 = input_shape[1], input_shape[2], input_shape[3], input_shape[4], input_shape[5], input_shape[6]
#         B = tf.shape(x)[0]

#         # O = kernel3d.shape[-1]
#         # h2 = kernel3d

#         # Step 1: Convolve over (N1, N2, N3)
#         x1 = tf.reshape(x, [-1, N4, N5, N6, self.inchannels])  # shape: (B*N3*N4, N1, N2, C)
#         # x1 = tf.reshape(x, [B * N4 * N5 * N6, N1, N2, N3, self.inchannels])
#         # print('x', x.shape)
#         # print(x1.shape)
#         y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
#         # print(y1.shape)
#         y1 = tf.reshape(y1, [B, N1, N2, N3, N4, N5, N6, self.inchannels])
#         # print(y1.shape)
#         y1 = tf.transpose(y1, perm=[0,4,5,6,1,2,3,7])
    
#         # Step 2: Convolve over (N4, N5, N6)
#         x2 = tf.reshape(y1, [-1, N1, N2, N3, self.filters])  # shape: (B*N1*N2, N3, N4, C) ## OK
#         # x2 = tf.reshape(y1, [B * N1 * N2 * N3, N4, N5, N6, self.inchannels])
#         # print(x2.shape)
#         y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
#         # print(y2.shape)
#         y2 = tf.reshape(y2, [B, N4, N5, N6, N1, N2, N3, self.inchannels])
#         # print(y2.shape)
#         y2 = tf.transpose(y2, perm=[0,4,5,6,1,2,3,7])
#         return y2
        

#     # # Function: 4D separable convolution using a single 2D kernel
#     # def __separable_conv4d(self, x):
#     #     # x: shape [B, N1, N2, N3, N4, C]
#     #     # B, N1, N2, N3, N4, C = x.shape
#     #     # Get static shape for dimensions that shouldn't change
#     #     input_shape = x.shape.as_list()
#     #     N1, N2, N3, N4, N3,  = input_shape[1], input_shape[2], input_shape[3], input_shape[4]
        
#     #     # Get dynamic batch size
#     #     B = tf.shape(x)[0]

#     #     ## Step 1: Convolve over (N1, N2)
#     #     x1 = tf.reshape(x, [-1, N1, N2, self.filters])  # shape: (B*N3*N4, N1, N2, C)
#     #     y1 = tf.nn.convolution(x1, self.kernel, padding='SAME')
#     #     y1 = tf.reshape(y1, [B, N1, N2, N3, N4, self.filters])   # (B, N1, N2, N3, N4, C)
#     #     y1 = tf.transpose(y1, perm=[0,3,4,1,2,5])
    
        
#     #     # print('+y1 ', y1.shape)


#     #     ## Step 2: Convolve over (N3, N4)
#     #     x2 = tf.reshape(y1, [-1, N3, N4, self.filters])  # shape: (B*N1*N2, N3, N4, C)
#     #     y2 = tf.nn.convolution(x2, self.kernel, padding='SAME')
#     #     y2 = tf.reshape(y2, [B, N1, N2, N3, N4, self.filters])    # final shape
#     #     y2 = tf.transpose(y2, perm=[0,3,4,1,2,5])

#     #     return y2
        
#     def get_kernel(self):
#         """Returns the kernel weights as a numpy array"""
#         return self.kernel.numpy()

#     # def compute_output_shape(self, input_shape):
#     #     return (input_shape[0], input_shape[1], input_shape[2], input_shape[3], input_shape[4], self.filters)
    
#     def get_config(self):
#         config = super().get_config()
#         config.update({
#             'filters': self.filters,
#             'kernel_size': self.kernel_size
#         })
#         return config

# # Create a model using the layer
# inputs = tf.keras.Input(shape=(16, 16, 16, 16, 16, 16, 2))  # (N1, N2, N3, N4, C)
# # inputs = tf.keras.Input(shape=(16, 16, 16, 16, 16, 16, 2))  # (N1, N2, N3, N4, C)
# outputs = SeparableConv6D(filters=6, kernel_size=3)(inputs)
# model = tf.keras.Model(inputs=inputs, outputs=outputs)
# model.summary()

# # # Test with random data
# test_input = tf.random.normal([2, 16, 16, 16, 16, 16, 16, 2])
# output = model(test_input)
# print(output.shape)
# del outputs, model, inputs, test_input, output

In [ ]:
# Input dimensions
B, N1, N2, N3, N4, N5, N6, C = 2, 8, 8, 8, 8, 8, 8, 3  # Batch size, 4D volume size, channels
# B, N1, N2, N3, N4, C = 1, 2, 2, 2, 2, 1  # Batch size, 4D volume size, channels
# B, N1, N2, N3, N4, C = 1, 3, 3, 2, 2, 1  # Batch size, 4D volume size, channels
x = tf.random.normal((B, N1, N2, N3, N4, N5, N6, C))
x.shape


TensorShape([2, 8, 8, 8, 8, 8, 8, 3])

In [4]:
layer = SeparableConv6D(filters=x.shape[-1], kernel_size=3)
yout = layer(x)
kernel3d = layer.get_kernel()
print("Kernel shape:", kernel3d.shape)  # Should be (3, 3, 2, 32)
yout.shape

Kernel shape: (3, 3, 3, 3, 3)


TensorShape([2, 8, 8, 8, 8, 8, 8, 3])

In [5]:
# # Shared 2D convolution kernel (C -> C to keep channels same)
# kH, kW = 3, 3
# kH, kW = 2, 2
# kernel2d = tf.random.normal((kH, kW, C, C))  # single 2D kernel reused



# Run the separable 4D convolution
y = separable_conv6d(x, kernel3d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel3d.shape)
print("Output shape:", y.shape)


NameError: name 'separable_conv6d' is not defined

In [467]:
# # Compare
diff = tf.reduce_max(tf.abs(y - yout))
print("Max absolute difference:", diff.numpy())
print("Outputs match:", tf.reduce_all(tf.abs(y - yout) < 1e-4).numpy())

Max absolute difference: 0.0
Outputs match: True


### separable_conv6d main

In [463]:
import tensorflow as tf

def separable_conv6d(x, kernel3d):
    # x: [B, N1, N2, N3, N4, N5, N6, C]
    B, N1, N2, N3, N4, N5, N6, C = x.shape
    # O = kernel3d.shape[-1]
    h2 = kernel3d

    # Step 1: Convolve over (N1, N2, N3)
    x1 = tf.reshape(x, [B * N4 * N5 * N6, N1, N2, N3, C])
    print('x', x.shape)
    print(x1.shape)
    y1 = tf.nn.convolution(x1, h2, padding='SAME')
    print(y1.shape)
    y1 = tf.reshape(y1, [B, N1, N2, N3, N4, N5, N6, C])
    print(y1.shape)
    y1 = tf.transpose(y1, perm=[0,4,5,6,1,2,3,7])
   
    # Step 2: Convolve over (N4, N5, N6)
    x2 = tf.reshape(y1, [B * N1 * N2 * N3, N4, N5, N6, C])
    print(x2.shape)
    y2 = tf.nn.convolution(x2, h2, padding='SAME')
    print(y2.shape)
    y2 = tf.reshape(y2, [B, N1, N2, N3, N4, N5, N6, C])
    print(y2.shape)
    y2 = tf.transpose(y2, perm=[0,4,5,6,1,2,3,7])

    return y2

# -------------------------------
# Test the function with dummy data
# -------------------------------

# Input shape
B, N1, N2, N3, N4, N5, N6, C = 1, 2, 2, 2, 2, 2, 2, 1
# B, N1, N2, N3, N4, N5, N6, C = 2, 2, 2, 2, 2, 2, 2, 2
B, N1, N2, N3, N4, N5, N6, C = 5, 2, 3, 4, 5, 6, 7, 8
# B, N1, N2, N3, N4, N5, N6, C = 5, 8, 8, 8, 8, 8, 8, 2

# O = 4
x = tf.random.normal((B, N1, N2, N3, N4, N5, N6, C))

# Shared 2D convolution kernel (C -> C to keep channels same)
kH, kW, kD = 3, 3, 3
kH, kW, kD = 2, 2, 2
kernel3d = tf.random.normal((kH, kW, kD, C, C))  # single 2D kernel reused

# Run the separable 4D convolution
y = separable_conv6d(x, kernel3d)

# Print shapes
print("Input shape :", x.shape)
print("Kernel 2D shape:", kernel3d.shape)
print("Output shape:", y.shape)



x (5, 2, 3, 4, 5, 6, 7, 8)
(1050, 2, 3, 4, 8)
(1050, 2, 3, 4, 8)
(5, 2, 3, 4, 5, 6, 7, 8)
(120, 5, 6, 7, 8)
(120, 5, 6, 7, 8)
(5, 2, 3, 4, 5, 6, 7, 8)
Input shape : (5, 2, 3, 4, 5, 6, 7, 8)
Kernel 2D shape: (2, 2, 2, 8, 8)
Output shape: (5, 5, 6, 7, 2, 3, 4, 8)


In [71]:
# Input shape
B, N1, N2, N3, N4, N5, N6, C = 1, 2, 2, 2, 2, 2, 2, 1
B, N1, N2, N3, N4, N5, N6, C = 2, 2, 2, 2, 2, 2, 2, 2
B, N1, N2, N3, N4, N5, N6, C = 5, 2, 3, 4, 5, 6, 7, 8
# B, N1, N2, N3, N4, N5, N6, C = 5, 8, 8, 8, 8, 8, 8, 1

O = 16
x = tf.random.normal((B, N1, N2, N3, N4, N5, N6, C))

# Shared 2D convolution kernel (C -> C to keep channels same)
kH, kW, kD = 3, 3, 3
kH, kW, kD = 2, 2, 2
kernel3d = tf.random.normal((kH, kW, kD, C, O))  # single 2D kernel reused



# x: [B, N1, N2, N3, N4, N5, N6, C]
B, N1, N2, N3, N4, N5, N6, C = x.shape
O = kernel3d.shape[-1]
h2 = kernel3d

# Step 1: Convolve over (N1, N2, N3)
x1 = tf.reshape(x, [B * N4 * N5 * N6, N1, N2, N3, C])
print(x.shape)
print(x1.shape)
y1 = tf.nn.convolution(x1, h2, padding='SAME')
print(y1.shape)
y1 = tf.reshape(y1, [B, N1, N2, N3, N4, N5, N6, O])
print(y1.shape)

# Step 2: Convolve over (N4, N5, N6)
x2 = tf.reshape(y1, [B * N1 * N2 * N3, N4, N5, N6, O])
print('x2    ', x2.shape)
print('filter', h2.shape)
y2 = tf.nn.convolution(x2, h2, padding='SAME')
print(y2.shape)
y2 = tf.reshape(y2, [B, N1, N2, N3, N4, N5, N6, O])
print(y2.shape)

(5, 2, 3, 4, 5, 6, 7, 8)
(1050, 2, 3, 4, 8)
(1050, 2, 3, 4, 16)
(5, 2, 3, 4, 5, 6, 7, 16)
x2     (120, 5, 6, 7, 16)
filter (2, 2, 2, 8, 16)
(120, 5, 6, 7, 16)
(5, 2, 3, 4, 5, 6, 7, 16)
